# Loss sensitivities over the globe: with and without the latent skip

Where in the processor's latent space and in the model input does the globe-wide loss
(`ObservationMaskedMSE` over every target observation) draw its information from, with and
without the latent skip? Both checkpoints are scored on the same date.

Both checkpoints have the same task and inputs: static forcings enter once, at t0, and the
time-varying forcings also at the target time, as a fourth input step (input hour +24) that
carries no observations. They differ only in `latent_skip`:
- `skip` is `../update-forcings-aa/runs/v0`, with the latent skip
- `no_skip` is this experiment's `runs/v0`, without it

The gradients are computed by `sensitivity.py` on a GPU, in `nese-anemoi` (`sbatch submit_sensitivity.sh`),
and written to `sensitivity/global/<t0>/<label>.zarr`. This notebook reads one date, set by `DATE`
(or `SENSITIVITY_DATE`), and runs in `nese`. Loading and drawing is in `sensitivity_plots.py`.

The gradients:
- **latent** is `|dL/dz|`, the norm over channels at each hidden (hpx6) node, where `z` is the processor input.
- **decoder** is `|g|`, the gradient at the decoder's hidden-node input. With `latent_skip` (`skip`) the decoder sees `processor(z) + z`, so `g` is the skip path alone and latent is `J_procᵀ g + g`. Without it (`no_skip`) the decoder sees `processor(z)`, so `g` is the gradient at the processor's output.
- **processor** is `|J_procᵀ g|`, what reached `z` through the processor: `latent − g` for `skip`, and latent itself for `no_skip`.
- **input** is `|dL/dx|`, summed over the variables of each kind (all sla missions, ...). `x` is the normalised, imputed input at each input hour.
- **encoder input** is the same at the encoder's input: each field's FieldEncoder latent (8 channels, at every cell, observed or not) and the passthrough inputs.

The magnitudes section compares absolute sensitivities. The maps after it are `log10` of the
sensitivity over its maximum, clipped at `LOG_FLOOR`, so the checkpoints are compared by shape.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from sensitivity_plots import NOISE_FLOOR
from sensitivity_plots import draw
from sensitivity_plots import draw_seen
from sensitivity_plots import draw_solid
from sensitivity_plots import load
from sensitivity_plots import map_axes
from sensitivity_plots import normalised_log
from sensitivity_plots import plot_shares

# t0 of the run of sensitivity.py to read; submit_sensitivity_notebook.sh sets it per date
DATE = os.environ.get("SENSITIVITY_DATE", "2020-09-13T12")
CHECKPOINTS = ["skip", "no_skip"]
SCALARS = ["base"]

In [ ]:
sens = load(f"global/{DATE}", CHECKPOINTS)
print(f"t0 {sens.attrs['t0']} -> target {sens.attrs['target_time']}")
sens[["loss", "loss_scale"]].to_dataframe()

## Sensitivity magnitudes

Absolute sensitivities, not shares. Left, the total along each path into the processor, summed
over hidden nodes. Right, the model-input sensitivity summed over cells, by kind and input hour, on
a log scale; count kinds are left out, since their gradient is zero by construction. The table gives
the totals and their ratio, no_skip over skip.

In [ ]:
paths = {"latent_sensitivity": "total", "decoder_sensitivity": "decoder", "processor_sensitivity": "processor"}
kinds = ["sla", "ugxt", "vgxt", "swh", "time_forcing", "static"]

for scalar in SCALARS:
    sensitivity = sens.sel(scalar=scalar)
    fig, (left, right) = plt.subplots(1, 2, figsize=(16, 3.5), width_ratios=[1, 3], constrained_layout=True)
    latent = xr.Dataset({title: sensitivity[name].sum("hidden") for name, title in paths.items()})
    latent.to_dataframe()[list(paths.values())].plot.bar(ax=left, logy=True)
    left.set(ylabel="sum over hidden nodes", title=f"{scalar}: processor input")
    by_kind = sensitivity["input_sensitivity"].sel(kind=kinds).sum("cell", min_count=1)
    rows = by_kind.stack(row=("checkpoint", "input_hour")).transpose("row", "kind").to_pandas()
    rows.dropna(how="all").plot.bar(ax=right, logy=True)
    right.set(ylabel="sum over cells", title=f"{scalar}: model input")
    right.legend(fontsize=7, ncol=3)

    totals = xr.Dataset(
        {**{title: sensitivity[name].sum("hidden") for name, title in paths.items()}, "input": sensitivity["input_total"].sum("cell")}
    ).to_dataframe()[[*paths.values(), "input"]]
    totals.loc["no_skip / skip"] = totals.loc["no_skip"] / totals.loc["skip"]
    display(totals)

## Sensitivity share by input type

Left sums every variable of a kind. Middle divides by the kind's number of input variables (22
per mission kind, 9 static, 3 time-varying forcing). Count columns enter only through the observed
mask (`counts > 0`), so their gradient is zero. Right is taken at the encoder's input instead:
each field's FieldEncoder latent (8 channels) and the passthrough inputs (1 channel per
variable), divided by channel count. All gradients are with respect to normalised inputs, so the
normalisations also enter the comparison.

In [ ]:
for scalar in SCALARS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 3.5), constrained_layout=True)
    plot_shares(axes, sens.sel(scalar=scalar))
    fig.suptitle(scalar)

## Processor input

Total, decoder input and processor path, each relative to the total's maximum, so the share of
each path is visible. For `no_skip`, processor is the total, and decoder is at the processor's
output rather than on a skip path.

In [ ]:
paths = {"latent_sensitivity": "total", "decoder_sensitivity": "decoder", "processor_sensitivity": "processor"}

for scalar in SCALARS:
    sensitivity = sens.sel(scalar=scalar)
    reference = sensitivity["latent_sensitivity"].max("hidden")
    fig, axes = map_axes(len(CHECKPOINTS), len(paths))
    for row, label in enumerate(CHECKPOINTS):
        for column, (name, title) in enumerate(paths.items()):
            values = np.log10(sensitivity[name] / reference).clip(min=-4).sel(checkpoint=label)
            image = draw(axes[row, column], values)
            axes[row, column].set_title(f"{scalar}, {label}: {title}")
    fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|grad| / max total)")

## Model input by input hour

Summed over all variables. Grey dots are cells where sla was observed in that input bin.

In [ ]:
for scalar in SCALARS:
    total = normalised_log(sens["input_sensitivity"].sel(scalar=scalar).sum("kind"), ["cell", "input_hour"])
    fig, axes = map_axes(len(CHECKPOINTS), len(sens["input_hour"]))
    for row, label in enumerate(CHECKPOINTS):
        for column, hour in enumerate(sens["input_hour"].values):
            values = total.sel(checkpoint=label, input_hour=hour)
            axes[row, column].set_title(f"{scalar}, {label}: input hour {hour}")
            if values.isnull().all():  # a checkpoint without this input step
                axes[row, column].set_visible(False)
                continue
            image = draw(axes[row, column], values)
            draw_seen(axes[row, column], sens["sla_seen"].sel(input_hour=hour))
    fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|dL/dx| / max)")

## Model input by kind, t0 (input hour 0)

Hour 0 rather than the last input hour, which is the forcing-only step.
Count kinds are left out: their gradient is zero by construction.

In [ ]:
for scalar in SCALARS:
    last = sens["input_sensitivity"].sel(scalar=scalar, input_hour=0, kind=kinds)
    last = normalised_log(last, "cell")
    fig, axes = map_axes(len(CHECKPOINTS), len(kinds))
    for row, label in enumerate(CHECKPOINTS):
        for column, kind in enumerate(kinds):
            image = draw(axes[row, column], last.sel(checkpoint=label, kind=kind))
            axes[row, column].set_title(f"{scalar}, {label}: {kind}")
    fig.colorbar(image, ax=axes, shrink=0.6, label="log10(|dL/dx| / max)")

## Where removing the latent skip moved the sensitivity

The ratio of the shape-normalised input sensitivity, no_skip over skip, summed over
kinds and input hours. Above 0 means a larger share of the sensitivity sits there without the skip. Cells below `NOISE_FLOOR` of the maximum in either run, including exact zeros, are
mid grey, since their ratio could be numerical.

In [ ]:
shape = sens["input_total"] / sens["input_total"].sum("cell")
reference, others = CHECKPOINTS[0], CHECKPOINTS[1:]
ratio = np.log10(shape.sel(checkpoint=others) / shape.sel(checkpoint=reference))
amplitude = sens["input_total"] / sens["input_total"].max("cell")
noisy = (amplitude.sel(checkpoint=others) < NOISE_FLOOR) | (amplitude.sel(checkpoint=reference) < NOISE_FLOOR)

fig, axes = map_axes(len(others), len(SCALARS))
for row, label in enumerate(others):
    for column, scalar in enumerate(SCALARS):
        values, noise = ratio.sel(checkpoint=label, scalar=scalar), noisy.sel(checkpoint=label, scalar=scalar)
        image = draw(axes[row, column], values.where(~noise, drop=True), cmap="RdBu_r", vmin=-1, vmax=1)
        draw_solid(axes[row, column], noise.where(noise, drop=True), color="0.55")
        axes[row, column].set_title(f"{scalar}: log10({label} / {reference})")
fig.colorbar(image, ax=axes, shrink=0.6)